In [ ]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [ ]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

len(documents)
print(documents)

In [ ]:
#Q2

from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [ ]:
question = "How does the agentic loop keep calling the model until it stops?"
index.search(
    question,
)

In [ ]:
from rag_helper_homework import RAGHomework
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI

openai_client = openai_client = OpenAI()

In [ ]:
assistant = RAGHomework(
    index,
    openai_client
)

answer, input_tokens = assistant.rag("How does the agentic loop keep calling the model until it stops?")

print(answer)
print(input_tokens)

In [ ]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [ ]:
len(chunks)

In [ ]:
index2 = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index2.fit(chunks)

assistant2 = RAGHomework(
    index2,
    openai_client
)

In [ ]:
answer2, input_tokens2 = assistant2.rag("How does the agentic loop keep calling the model until it stops?")

print(answer2)
print(input_tokens2)

In [ ]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [ ]:
def search(query: str) -> dict[str, str]:
    """
    Search the lessons to answer questions
    """
    return index.search(
        query,
        num_results=5,
    )

agent_tools = Tools()
agent_tools.add_tool(search)

In [ ]:
agent_tools.get_tools()

In [ ]:
instructions = """
You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.
""".strip()

In [ ]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [ ]:
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)